# Checking for Hallucinations using NLI

This example is taken from the short course on Deeplearning.ai platform.

Course details: **Safe and Reliable AI via Guardrails**
https://www.deeplearning.ai/short-courses/safe-and-reliable-ai-via-guardrails/


Start by setting up the notebook to minimize warnings:


In [5]:
# Warning control
import warnings
warnings.filterwarnings("ignore")

# Installation

In [7]:
!pip install guardrails-ai

INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-http to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.2/235.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 996.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Helper functions ofr Chat and RAG with a in-memory VectorDB

In [2]:
import os
from typing import List, Tuple

import ipywidgets as widgets
import numpy as np
from dotenv import find_dotenv, load_dotenv
from guardrails import Guard, settings
from IPython.display import display
from sentence_transformers import SentenceTransformer

# these expect to find a .env file at the directory above the lesson.
# the format for that file is (without the comment)#API_KEYNAME=AStringThatIsTheLongAPIKeyFromSomeService


def load_env():
    _ = load_dotenv(find_dotenv())


def get_openai_api_key():
    load_env()
    openai_api_key = os.getenv("OPENAI_API_KEY")
    return openai_api_key


def get_guardrails_api_key():
    load_env()
    guardrails_api_key = os.getenv("GUARDRAILS_API_KEY")
    return guardrails_api_key


class ChatWidget:
    def __init__(self, client=None, guard_name=None, system_message=None):
        """
        A widget for handling chat interactions.

        Parameters
        ----------
        client : object
            The OpenAI client object to use for generating responses.
        system_message : str, optional
            An optional system message to initialize the chat with.
        """
        self.chat_logs = []
        self.messages = []
        if system_message:
            self.messages.append({"role": "assistant", "content": system_message})
        # self.main_output = widgets.Output()

        self.text_input = widgets.Textarea(
            value="",
            placeholder="Type something then click the blue submit button.",
            disabled=False,
            continuous_update=False,
            layout=widgets.Layout(width="400px", height="75px"),
            form="chatform",
        )

        self.text_input.observe(self.handle_submit, names="value")
        self.submit_button = widgets.Button(
            form="chatform",
            icon="paper-plane",
            button_style="primary",
            type="submit",
            layout=widgets.Layout(width="40px", margin_y="auto"),
        )

        action_bar = widgets.HBox(
            [
                self.text_input,
                widgets.VBox(
                    [self.submit_button],
                    layout=widgets.Layout(justify_content="center", margin_y="auto"),
                ),
            ],
            layout=widgets.Layout(
                justify_content="center", width="480px", padding_y="10px"
            ),
        )

        self.chat_box = widgets.VBox(
            [], layout=widgets.Layout(max_height="300px", overflow_y="auto")
        )
        self.main_container = widgets.VBox(
            [self.chat_box, action_bar],
            layout=widgets.Layout(width="505px", justify_content="center"),
        )
        self._client = client
        self._guard_name = guard_name

    @property
    def client(self):
        return self._client

    @client.setter
    def client(self, value):
        self._client = value

    def reset(self):
        self.chat_logs = []
        self.messages = []
        self.chat_box.children = self.chat_logs

    def create_msg_widget(self, type, content, is_error=False):
        """Utility function to create a message widget based on the type"""
        common_style = """
            padding: 8px;
            margin: 2px 0;
            border-radius: 5px;
            width: fit-content;
            max-width: 70%;
            word-wrap: break-word;
            white-space: pre-wrap;
            overflow-wrap: break-word;
            line-height: 1.4;
        """

        if type == "user":
            style = (
                common_style
                + "justify-content: flex-end; background-color: #f0f0f0; float: right;"
            )
        elif type == "bot":
            style = common_style + "justify-content: flex-start;"
        else:
            raise ValueError("Type must be either 'user' or 'bot'")

        html_content = f'<div style="{style}">{content}</div>'
        return widgets.HTML(html_content)

    def update_chat_box(self, user_msg, bot_msg, error=False):
        user_widget = self.create_msg_widget("user", user_msg)
        bot_widget = self.create_msg_widget("bot", bot_msg, error)
        self.chat_logs.extend([user_widget, bot_widget])
        self.chat_box.children = self.chat_logs

    def show_loading(self, message):
        user_widget = self.create_msg_widget("user", message)
        loading_widget = self.create_msg_widget("bot", "Thinking...")
        loading_chat_logs = self.chat_logs.copy()
        loading_chat_logs.extend([user_widget, loading_widget])
        self.chat_box.children = loading_chat_logs

    def handle_submit(self, change):
        if (
            change["type"] == "change"
            and change["name"] == "value"
            and change["new"] != ""
        ):
            user_msg = change["new"]
            self.show_loading(user_msg)
            change["owner"].value = ""

            # self.remove_loading()
            query_messages = self.messages.copy()
            query_messages.append({"role": "user", "content": user_msg})

            # get the bot response
            error = False
            try:
                bot_message = self.bot_response_generator(query_messages)

                # write the user msg and bot response back in to message history
                self.messages.append({"role": "user", "content": user_msg})
                self.messages.append({"role": "assistant", "content": bot_message})

            except Exception as e:
                print(e)

                # we don't write here cuz it's errors
                bot_message = str(e)
                error = True

            # Clear the input after submission
            self.update_chat_box(user_msg, bot_message, error)

    def display(self):
        display(self.main_container)

    def bot_response_generator(self, message_history):
        if self.client:
            response = self.client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=message_history,
                seed=42,
                temperature=0.0,
            )
            bot_msg = response.choices[0].message.content
            return bot_msg
        else:
            settings.use_server = True
            response = Guard(name=self._guard_name)(
                model="gpt-3.5-turbo",
                messages=message_history,
            )

            settings.use_server = False
            return response.validated_output


def chunk_markdown_files(directory):
    chunks = []

    for filename in os.listdir(directory):
        if filename.endswith(".md"):
            file_path = os.path.join(directory, filename)
            with open(file_path, "r", encoding="utf-8") as file:
                content = file.read()

            # Split content into lines
            lines = content.split("\n")

            title = os.path.splitext(filename)[0]
            current_h1 = ""
            current_h2 = ""
            current_content = []

            for line in lines:
                if line.startswith("# "):
                    # New h1 header
                    if current_content:
                        chunks.append(
                            format_chunk(title, current_h1, current_h2, current_content)
                        )
                    current_h1 = line[2:].strip()
                    current_h2 = ""
                    current_content = []
                elif line.startswith("## "):
                    # New h2 header
                    if current_content:
                        chunks.append(
                            format_chunk(title, current_h1, current_h2, current_content)
                        )
                    current_h2 = line[3:].strip()
                    current_content = []
                else:
                    # Content (including h3 headers and list items)
                    current_content.append(line)

            # Add the last chunk
            if current_content:
                chunks.append(
                    format_chunk(title, current_h1, current_h2, current_content)
                )

    return chunks


def format_chunk(title, h1, h2, content):
    section_info = f"{h1}/{h2}" if h2 else h1
    content_text = "\n".join(content).strip()
    return f"Title: {title}\nSection: {section_info}\n{content_text}"


class SimpleVectorDB:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.embeddings = []
        self.strings = []

    def add_strings(self, strings: List[str]):
        new_embeddings = self.model.encode(strings)
        self.embeddings.extend(new_embeddings)
        self.strings.extend(strings)

    def query(
        self, query_string: str, k: int, threshold: float
    ) -> List[Tuple[str, float]]:
        query_embedding = self.model.encode([query_string])[0]

        if not self.embeddings:
            return []

        embeddings_array = np.array(self.embeddings)

        # Calculate cosine similarities
        similarities = np.dot(embeddings_array, query_embedding) / (
            np.linalg.norm(embeddings_array, axis=1) * np.linalg.norm(query_embedding)
        )

        # Convert similarities to distances (1 - similarity)
        distances = 1 - similarities
        # distances = similarities

        # Sort indices by distance
        sorted_indices = np.argsort(distances)

        results = []
        for idx in sorted_indices:
            if distances[idx] < threshold and len(results) < k:
                results.append((self.strings[idx], float(distances[idx])))
            else:
                break

        results.reverse()

        return results

    @classmethod
    def from_files(cls, directory: str):
        chunks = chunk_markdown_files(directory)
        db = cls()
        db.add_strings(chunks)
        return db


class RAGChatWidget(ChatWidget):
    def __init__(
        self,
        client=None,
        guard_name=None,
        system_message=None,
        vector_db=None,
        # data_directory="shared_data/",
    ):
        super().__init__(
            client=client, guard_name=guard_name, system_message=system_message
        )
        self.vector_db = vector_db
        # self.data_directory = data_directory

        # self.hydrate_vector_db()

    def hydrate_vector_db(self):
        chunks = chunk_markdown_files(self.data_directory)
        self.vector_db.add_strings(chunks)

    def bot_response_generator(self, message_history, context=None):
        if self.client:
            response = self.client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=message_history,
                seed=42,
                temperature=0.0,
            )
            bot_msg = response.choices[0].message.content
            return bot_msg
        else:
            # Context is a list of touples, we want to map down to the first value in the tuples
            sources = [c[0] for c in context]
            settings.use_server = True

            response = Guard(name=self._guard_name)(
                model="gpt-3.5-turbo",
                messages=message_history,
                metadata={"sources": sources, "chunk_strategy": "sentence"},
            )

            settings.use_server = False

            return response.validated_output

    def handle_submit(self, change):
        if (
            change["type"] == "change"
            and change["name"] == "value"
            and change["new"] != ""
        ):
            # extract message user sent
            user_msg = change["new"]
            self.show_loading(user_msg)
            # Clear the input after submission
            change["owner"].value = ""

            context = self.retrieve(user_msg, k=3)

            # do retrieval, add to message history
            augmented_user_msg = self.retrieval_augmentation(user_msg, context)
            error = False
            query_messages = self.messages.copy()
            query_messages.append({"role": "user", "content": augmented_user_msg})
            # get the bot response
            try:
                bot_message = self.bot_response_generator(
                    query_messages, context=context
                )

                # write the user msg and bot response back in to message history
                self.messages.append({"role": "user", "content": augmented_user_msg})
                self.messages.append({"role": "assistant", "content": bot_message})

            except Exception as e:
                print(e)

                # we don't write here cuz it's errors
                bot_message = e.body['detail']
                error = True

            # We show user_msg here instead of the augmented_user_msg to hide the retrieval
            self.update_chat_box(user_msg, bot_message, error)

    def retrieve(self, user_msg, k=1, threshold=0.9):
        retrieval = self.vector_db.query(user_msg, k=k, threshold=threshold)
        retrieved_ctx = ""
        for idx, (ctx, _) in enumerate(retrieval):
            retrieved_ctx += f"# Context {idx + 1}:\n{ctx}\n\n"
        # return retrieval
        return retrieved_ctx

    def retrieval_augmentation(self, user_msg, retrieval):
        augmented_user_msg = f"""\n
Use this context to help answer the question:

{retrieval}

User message:
{user_msg}
"""

        return augmented_user_msg


# # Example usage
# if __name__ == "__main__":
#     directory = "shared_data/"
#     chunks = chunk_markdown_files(directory)
#     for chunk in chunks:
#         print(chunk)
#         print("-" * 50)


Import OpenAI client and helpers to set up RAG chatbot and vector database:


In [4]:
from openai import OpenAI
# from helper import RAGChatWidget, SimpleVectorDB

# Set up the client, vector database, and system message for the chatbot:


In [5]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

In [11]:
# Setup an OpenAI client
unguarded_client = OpenAI()

# Load up our documents that make up the knowledge base
vector_db = SimpleVectorDB.from_files("shared_data/")

# Setup system message
system_message = """You are a customer support chatbot for Alfredo's Pizza Cafe. Your responses should be based solely on the provided information.

Here are your instructions:

### Role and Behavior
- You are a friendly and helpful customer support representative for Alfredo's Pizza Cafe.
- Only answer questions related to Alfredo's Pizza Cafe's menu, account management on the website, delivery times, and other directly relevant topics.
- Do not discuss other pizza chains or restaurants.
- Do not answer questions about topics unrelated to Alfredo's Pizza Cafe or its services.

### Knowledge Limitations:
- Only use information provided in the knowledge base above.
- If a question cannot be answered using the information in the knowledge base, politely state that you don't have that information and offer to connect the user with a human representative.
- Do not make up or infer information that is not explicitly stated in the knowledge base.
"""



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Initialize the chatbot using the settings above:


In [12]:
# Setup RAG chatbot
rag_chatbot = RAGChatWidget(
    client=unguarded_client,
    system_message=system_message,
    vector_db=vector_db,
)

To revisit the hallucination example from Lesson 1, run the cell below to open the chatbot then paste in the prompt:


In [13]:
rag_chatbot.display()

# Copy and paste this prompt into the chatbot above:
"""
how do i reproduce your veggie supreme pizza on my own? can you share detailed instructions?
"""


'\nhow do i reproduce your veggie supreme pizza on my own? can you share detailed instructions?\n'

## Setup an Natural Language Inference (NLI) Model

Import some additional packages to setup the NLI model:


In [14]:
# Type hints
from typing import Dict, List, Optional

# Standard ML libraries
import numpy as np
import nltk
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Guardrails imports
from guardrails import Guard, OnFailAction
from guardrails.validator_base import (
    FailResult,
    PassResult,
    ValidationResult,
    Validator,
    register_validator,
)


Create a hugging face pipeline to access the NLI model (**Note:** the weights will take about 30 seconds to download):

In [15]:
entailment_model = 'GuardrailsAI/finetuned_nli_provenance'
NLI_PIPELINE = pipeline("text-classification", model=entailment_model)

config.json:   0%|          | 0.00/934 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

Try out the pipeline:

# Example 1: Entailed sentence

In [16]:
# Example 1: Entailed sentence
premise = "The sun rises in the east and sets in the west."
hypothesis = "The sun rises in the east."
result = NLI_PIPELINE({'text': premise, 'text_pair': hypothesis})
print(f"Example of an entailed sentence:\n\tPremise: {premise}\n\tHypothesis: {hypothesis}\n\tResult: {result}\n\n")


Example of an entailed sentence:
	Premise: The sun rises in the east and sets in the west.
	Hypothesis: The sun rises in the east.
	Result: {'label': 'entailment', 'score': 0.8697451949119568}




# Example 2: Contradictory sentence

In [ ]:
# Example 2: Contradictory sentence
premise = "The sun rises in the east and sets in the west."
hypothesis = "The sun rises in the west."
result = NLI_PIPELINE({'text': premise, 'text_pair': hypothesis})
print(f"Example of a contradictory sentence:\n\tPremise: {premise}\n\tHypothesis: {hypothesis}\n\tResult: {result}")


## Building a Hallucination Validator

In this section, you'll build a validator to test for hallucinations in the responses of your RAG chatbot. The validator will check that the response is grounded in the texts of your vector database.

Start by setting up a validator with stubs for the `__init__` and `validate` functions:


In [17]:
@register_validator(name="hallucination_detector", data_type="string")
class HallucinationValidation(Validator):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def validate(
        self, value: str, metadata: Optional[Dict[str, str]] = None
    ) -> ValidationResult:
        pass


Next, start fleshing out the pieces of the validator. Start by building the function that will split the response of the LLM into individual sentences:


In [18]:
@register_validator(name="hallucination_detector", data_type="string")
class HallucinationValidation(Validator):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def validate(
        self, value: str, metadata: Optional[Dict[str, str]] = None
    ) -> ValidationResult:
        # Split the text into sentences
        sentences = self.split_sentences(value)
        pass

    def split_sentences(self, text: str) -> List[str]:
        if nltk is None:
            raise ImportError(
                "This validator requires the `nltk` package. "
                "Install it with `pip install nltk`, and try again."
            )

        return nltk.sent_tokenize(text)


Now finalize the logic of the validate function. You'll loop through each sentence and check if it is grounded in the texts in the vector database using the `find_relevant_sources` and `check_entailment` functions. Then update the `__init__` function to set up the needed class variables.


In [19]:
@register_validator(name="hallucination_detector", data_type="string")
class HallucinationValidation(Validator):
    def __init__(
            self,
            embedding_model: Optional[str] = None,
            entailment_model: Optional[str] = None,
            sources: Optional[List[str]] = None,
            **kwargs
        ):
        if embedding_model is None:
            embedding_model = 'all-MiniLM-L6-v2'
        self.embedding_model = SentenceTransformer(embedding_model)

        self.sources = sources

        if entailment_model is None:
            entailment_model = 'GuardrailsAI/finetuned_nli_provenance'
        self.nli_pipeline = pipeline("text-classification", model=entailment_model)

        super().__init__(**kwargs)

    def validate(
        self, value: str, metadata: Optional[Dict[str, str]] = None
    ) -> ValidationResult:
        # Split the text into sentences
        sentences = self.split_sentences(value)

        # Find the relevant sources for each sentence
        relevant_sources = self.find_relevant_sources(sentences, self.sources)

        entailed_sentences = []
        hallucinated_sentences = []
        for sentence in sentences:
            # Check if the sentence is entailed by the sources
            is_entailed = self.check_entailment(sentence, relevant_sources)
            if not is_entailed:
                hallucinated_sentences.append(sentence)
            else:
                entailed_sentences.append(sentence)

        if len(hallucinated_sentences) > 0:
            return FailResult(
                error_message=f"The following sentences are hallucinated: {hallucinated_sentences}",
            )

        return PassResult()

    def split_sentences(self, text: str) -> List[str]:
        if nltk is None:
            raise ImportError(
                "This validator requires the `nltk` package. "
                "Install it with `pip install nltk`, and try again."
            )
        return nltk.sent_tokenize(text)

    def find_relevant_sources(self, sentences: str, sources: List[str]) -> List[str]:
        source_embeds = self.embedding_model.encode(sources)
        sentence_embeds = self.embedding_model.encode(sentences)

        relevant_sources = []

        for sentence_idx in range(len(sentences)):
            # Find the cosine similarity between the sentence and the sources
            sentence_embed = sentence_embeds[sentence_idx, :].reshape(1, -1)
            cos_similarities = np.sum(np.multiply(source_embeds, sentence_embed), axis=1)
            # Find the top 5 sources that are most relevant to the sentence that have a cosine similarity greater than 0.8
            top_sources = np.argsort(cos_similarities)[::-1][:5]
            top_sources = [i for i in top_sources if cos_similarities[i] > 0.8]

            # Return the sources that are most relevant to the sentence
            relevant_sources.extend([sources[i] for i in top_sources])

        return relevant_sources

    def check_entailment(self, sentence: str, sources: List[str]) -> bool:
        for source in sources:
            output = self.nli_pipeline({'text': source, 'text_pair': sentence})
            if output['label'] == 'entailment':
                return True
        return False


# Try out the validator.

First you'll create an instance of the `HallucinationValidation` class above, passing in the same sentence as you used in the pipeline test above:


In [20]:
hallucination_validator = HallucinationValidation(
    sources = ["The sun rises in the east and sets in the west"]
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Then use the `validate()` function of this object, passing in the sentence you want to test. The first example does not entail, but the second does:


In [22]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

# Hallucination - groundedness check

In [23]:
result = hallucination_validator.validate("The sun sets in the east")
print(f"Validation outcome: {result.outcome}")
if result.outcome == "fail":
    print(f"Error message: {result.error_message}")


Validation outcome: Outcome.FAIL
Error message: The following sentences are hallucinated: ['The sun sets in the east']


In [24]:
result = hallucination_validator.validate("The sun sets in the west")
print(f"Validation outcome: {result.outcome}")
if result.outcome == "fail":
    print(f"Error message: {result.error_message}")


Validation outcome: Outcome.PASS
